# Sense Check: Monthly Bond Panel

This notebook inspects the output of `03_01a_build_monthly_bond_panel.py`.
It reports schema, date coverage, missingness, and a few quick sanity checks.

In [2]:
from pathlib import Path
import pandas as pd

# Use the notebook folder as the base directory
notebook_dir = Path.cwd()
# The monthly panel is written to the local outputs directory
out_dir = notebook_dir / "outputs"

# Resolve the expected output file path
panel_path = out_dir / "bond_panel_monthly.parquet"
# Fail fast with a clear message if the file is missing
if not panel_path.exists():
    raise FileNotFoundError(
        f"Missing {panel_path}. Run 03_01_build_monthly_bond_panel.py first."
    )

# Load the panel using fastparquet (consistent with the builder script)
panel = pd.read_parquet(panel_path, engine="fastparquet")
# Preview a few rows for quick sanity check
panel.head()

,RIC,Trade Date,price,month_end,yield,price_return
0,001055AY8=,2018-10-31,97.549,2018-10-31,4.906,NaN
1,001055AY8=,2018-11-30,99.653,2018-11-30,4.772,0.021569
2,001055AY8=,2018-12-31,102.415,2018-12-31,4.600,0.027716
3,001055AY8=,2019-01-31,106.537,2019-01-31,4.354,0.040248
4,001055AY8=,2019-02-28,106.299,2019-02-28,4.367,-0.002234


In [3]:
# Inspect column dtypes and non-null counts
panel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102018 entries, 0 to 102017
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   RIC           102018 non-null  object        
 1   Trade Date    102018 non-null  datetime64[ns]
 2   price         101698 non-null  float64       
 3   month_end     102018 non-null  datetime64[ns]
 4   yield         100881 non-null  float64       
 5   price_return  100130 non-null  float64       
dtypes: datetime64[ns](2), float64(3), object(1)
memory usage: 4.7+ MB


In [4]:
# Coverage: date range, unique bonds, and total rows
panel["month_end"].min(), panel["month_end"].max(), panel["RIC"].nunique(), len(panel)

(Timestamp('2002-01-31 00:00:00'),
 Timestamp('2025-09-30 00:00:00'),
 1720,
 102018)

In [5]:
# Quick missingness rates (top 10)
panel.isna().mean().sort_values(ascending=False).head(10)

price_return    0.018507
yield           0.011145
price           0.003137
RIC             0.000000
Trade Date      0.000000
month_end       0.000000
dtype: float64

In [6]:
# Basic return sanity checks (distributional summary)
panel["price_return"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

count    100130.000000
mean         -0.000421
std           0.052207
min          -0.928989
1%           -0.088411
5%           -0.053703
50%           0.000354
95%           0.050405
99%           0.095481
max          12.585227
Name: price_return, dtype: float64

## Quick Summary (Sample Output)

- **Rows:** ~102k monthly observations
- **Bonds (RICs):** ~1,720
- **Date range:** 2002-01 to 2025-09 (month-end)
- **Missingness:** low (price ~0.3%, yield ~1.1%, returns ~1.9%)
- **Returns:** centered near zero with occasional extreme values (check outliers before modeling)